# Quill — Notebook 09: Capstone, End-to-End Cord Experiment

**Goal:** bring together Knot encoding, gates, superposition, controlled operations, measurement, and decoding in one small experiment.

We will use `Cord("HI")`: two character Knots, eight qubits per character. First, we will make one-bit superposition in the `H` Knot and use a CNOT to correlate that bit with a bit in the `I` Knot. Then we will measure and decode the observed strings.

> This is a teaching experiment, not a quantum-advantage claim. The characters are encoded as ASCII bit patterns; the circuit manipulates those bits, not the meanings of the letters.

## 1. The encoding convention

Each character is represented by its 8-bit ASCII value. For example,

$$
\mathrm{ASCII}(H)=72=(01001000)_2
$$

We display bytes most-significant bit first, but Qiskit qubit index `0` stores the least-significant bit. Keep that distinction in mind when selecting a qubit.

For this notebook, the `H` and `I` bytes are adjacent values:

$$
H=72=(01001000)_2,\qquad I=73=(01001001)_2
$$

They differ only in the least-significant bit. Applying an X gate to that bit changes the encoded value 72 to 73, so `H` decodes as `I`.

## 2. Install Qiskit (run once in Colab)

In [ ]:
!pip -q install qiskit qiskit-aer

## 3. Helpers: encode, build a Cord, and decode

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from collections import Counter

def encode_text(text):
    """Prepare a basis state containing each character's ASCII byte."""
    qc = QuantumCircuit(8 * len(text), 8 * len(text))
    for char_index, char in enumerate(text):
        value = ord(char)
        if value > 127:
            raise ValueError("This teaching notebook uses 7-bit ASCII characters only.")
        for bit in range(8):
            if (value >> bit) & 1:
                qc.x(8 * char_index + bit)
    return qc

def decode_bitstring(bitstring, n_chars):
    """Decode Qiskit's displayed measurement string into characters."""
    # Qiskit displays the highest classical bit on the left. Reverse to index bits.
    bits = bitstring.replace(" ", "")[::-1]
    chars = []
    for i in range(n_chars):
        value = sum(int(bits[8*i + b]) << b for b in range(8))
        chars.append(chr(value))
    return "".join(chars)

def run_counts(qc, shots=1000):
    measured = qc.copy()
    measured.measure(range(measured.num_qubits), range(measured.num_qubits))
    sim = AerSimulator()
    compiled = transpile(measured, sim)
    return sim.run(compiled, shots=shots).result().get_counts()

print("H:", ord("H"), format(ord("H"), "08b"))
print("I:", ord("I"), format(ord("I"), "08b"))

## 4. Experiment A — a deterministic character change

Prepare `Cord("HI")`, then apply X to qubit `0`, the least-significant bit of the first Knot.

For the first character:

$$
|H\rangle_K=|01001000\rangle
$$

Since the selected bit is `0`, the X gate flips it to `1`:

$$
X_0|01001000\rangle=|01001001\rangle=|I\rangle_K
$$

The gate does not understand the letter. It flips one encoded bit; decoding afterward happens to produce `I`.

In [ ]:
qc = encode_text("HI")
qc.x(0)  # LSB of the first character Knot
print(qc.draw())
counts = run_counts(qc, shots=100)
for bits, n in counts.items():
    print(decode_bitstring(bits, 2), n)

## 5. Experiment B — superposition plus a controlled operation

Now prepare `HI`, apply a Hadamard to qubit `0`, and use that qubit to control a CNOT targeting qubit `8` (the least-significant bit of the second Knot).

The first selected bit begins in `0`. The Hadamard creates

$$
H|0\rangle=\frac{|0\rangle+|1\rangle}{\sqrt{2}}
$$

The CNOT copies the control bit's computational-basis value into the target by flipping the target when the control is `1`. Since the target bit starts at `1` for `I`, the joint state becomes

$$
\frac{|H I\rangle+|I H\rangle}{\sqrt{2}}
$$

Here the notation abbreviates the full 16-qubit basis states after the selected bits have been changed. The two measured strings should be `HI` and `IH`, roughly half each in an ideal simulator. The correlations—not any single outcome—are the key observation.

In [ ]:
qc = encode_text("HI")
qc.h(0)
qc.cx(0, 8)
print(qc.draw())
counts = run_counts(qc, shots=1000)
summary = Counter()
for bits, n in counts.items():
    summary[decode_bitstring(bits, 2)] += n
summary.most_common()

## 6. Interpret the result carefully

- In Experiment A, the result is deterministic: the selected bit flips, and `H` decodes as `I`.
- In Experiment B, the Hadamard creates two possible branches and CNOT correlates selected bits across the two Knots.
- Measuring once returns one result, not a list of all possibilities.
- The displayed `HI` / `IH` shorthand refers to the decoded ASCII strings; it does not mean the circuit manipulates semantic letters.

A subtle point: this state is entangled across the selected qubits, but each complete decoded string is only a classical measurement outcome. To verify entanglement experimentally, study correlations across repeated shots and compare against a separable control circuit.

## 7. Exercises

1. Change the initial Cord to `"AB"`. Predict which character appears if you apply X to qubit `0`.
2. In Experiment B, change the target from qubit `8` to qubit `9`. Predict what correlation you expect before running it.
3. Remove the CNOT. How does the distribution of measured strings change?
4. Replace `qc.h(0)` with `qc.x(0)`. Is the result still probabilistic?

**Takeaway:** a Cord is an ordered collection of character registers; gates act on qubits inside those registers. The decoded text is an interpretation of measurement results.